In [ ]:
import os

from train_roberta_classifier import TRClassifier as TRClassifierRoberta
from train_gemma_classifier import TRClassifier as TRClassifierGemma


MODEL_CONFIGS = [
    {
        "model_name": "mbert",
        "checkpoint": '../../../models/mbert_tuned/checkpoint-5715',
        "batch_size": 64,
        "max_length": 512,
        "classifier_class": TRClassifierRoberta,
    },
    {
        "model_name": "xlmroberta",
        "checkpoint": '../../../models/xlmroberta_tuned/checkpoint-5715',
        "batch_size": 64,
        "max_length": 512,
        "classifier_class": TRClassifierRoberta,
    },
    {
        "model_name": "mbert_extended",
        "checkpoint": '../../../models/mbert_extended_tuned/checkpoint-5742/',
        "batch_size": 64,
        "max_length": 512,
        "classifier_class": TRClassifierRoberta,
    },
    {
        "model_name": "xlmroberta_extended",
        "checkpoint": "../../../models/xlmroberta_extended_tuned/checkpoint-5742/",
        "batch_size": 64,
        "max_length": 512,
        "classifier_class": TRClassifierRoberta,
    },
    {
        "model_name": "xlmroberta_ms",
        "checkpoint": "../../../models/xlmroberta_ms_tuned/checkpoint-2410/",
        "batch_size": 64,
        "max_length": 512,
        "classifier_class": TRClassifierRoberta,
    },
    {
        "model_name": "gemma3_1b",
        "checkpoint": "../../../models/gemma3_1b_merged/",
        "batch_size": 4,
        "max_length": 512,
        "classifier_class": TRClassifierGemma,
    },
    {
        "model_name": "gemma3_1b_extended",
        "checkpoint": "../../../models/gemma3_1b_extended_merged/",
        "batch_size": 4,
        "max_length": 512,
        "classifier_class": TRClassifierGemma,
    },
]

DATA_DIR = "../../../data"

TRAIN_DATA_PATH      = f"{DATA_DIR}/train_data.parquet"
TEST_DATA_PATH       = f"{DATA_DIR}/test_data.parquet"
MULTISOCIAL_PATH     = f"{DATA_DIR}/external_data/multisocial/multisocial_anonymized.csv"
AIGTBENCH_PATH       = f"{DATA_DIR}/external_data/AIGTBench"
FOX8_PATH            = f"{DATA_DIR}/external_data/fox8_23_dataset.ndjson"

AIGTBENCH_PLATFORM_FILTER = "reddit"

SCORES_DIR = "../../../scores"
os.makedirs(SCORES_DIR, exist_ok=True)
print(f"Scores will be saved to: {os.path.abspath(SCORES_DIR)}")

In [ ]:
import json
import joblib

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from train_roberta_classifier import TRClassifier

In [ ]:
def make_classifier(config: dict) -> TRClassifier:
    cls = config.get("classifier_class", TRClassifierRoberta)
    return cls({
        "initial_model": config["checkpoint"],
        "batch_size":    config["batch_size"],
        "model_name":    config["model_name"],
        "max_length":    config["max_length"],
        "n_epochs":      1,
    })



def predict_external(classifier: TRClassifier, texts: list, model_name: str) -> list:
    dataset = pd.DataFrame({
        "generated_text":          texts,
        "generated_text_language": ["placeholder"] * len(texts),
        "real_text":               ["placeholder"] * len(texts),
        "real_text_language":      ["placeholder"] * len(texts),
    })
    scored = classifier.predict(dataset)
    return scored[f"tc_{model_name}_generated_score"].tolist()


def extract_pair_scores(scored_df: pd.DataFrame, model_name: str) -> dict:
    real_col = f"tc_{model_name}_real_score"
    gen_col  = f"tc_{model_name}_generated_score"
    scores = {}
    for _, row in scored_df.iterrows():
        key = (row["model"], row["pipeline"], row["user_id"], row["thread_id"])
        scores[key] = (row[real_col], row[gen_col])
    return scores

In [ ]:
train = pd.read_parquet(TRAIN_DATA_PATH)
test  = pd.read_parquet(TEST_DATA_PATH)
val   = train[train.is_val == True].copy()

print(f"Train rows:     {len(train):,}")
print(f"Val rows:       {len(val):,}")
print(f"Test rows:      {len(test):,}")

In [ ]:
multisocial = pd.read_csv(MULTISOCIAL_PATH)
multisocial = multisocial[multisocial.split == "test"].copy()
print(f"Multisocial test rows: {len(multisocial):,}")
multisocial.head(2)

In [ ]:
aigtbench = pd.read_parquet(AIGTBENCH_PATH)
if AIGTBENCH_PLATFORM_FILTER:
    aigtbench = aigtbench[aigtbench.social_media_platform == AIGTBENCH_PLATFORM_FILTER].copy()
print(f"AIGTBench rows (filter={AIGTBENCH_PLATFORM_FILTER!r}): {len(aigtbench):,}")
aigtbench.head(2)

In [ ]:
with open(FOX8_PATH, "r", encoding="utf-8") as f:
    fox8_raw = [json.loads(line) for line in f]

fox8_rows = []
for sample in tqdm(fox8_raw, desc="Parsing fox8"):
    label   = 0 if sample["label"] == "human" else 1
    user_id = sample["user_id"]
    for tweet in sample["user_tweets"]:
        fox8_rows.append({"text": tweet["text"], "user_id": user_id, "label": label})

fox8 = pd.DataFrame(fox8_rows)
print(f"Fox8 rows: {len(fox8):,}  |  users: {fox8.user_id.nunique():,}")
fox8.head(2)

In [ ]:
for config in MODEL_CONFIGS:
    model_name = config["model_name"]
    print(f"\n{'='*60}")
    print(f"Model: {model_name}  |  checkpoint: {config['checkpoint']}")
    print(f"{'='*60}")

    classifier = make_classifier(config)

    print("Scoring test set...")
    test_scored = classifier.predict(test.copy())
    test_scores = extract_pair_scores(test_scored, model_name)
    joblib.dump(test_scores, f"{SCORES_DIR}/{model_name}_test_scores.joblib")
    print(f"  Saved {len(test_scores):,} test score pairs.")

    print("Scoring validation set...")
    val_scored  = classifier.predict(val.copy())
    val_scores  = extract_pair_scores(val_scored, model_name)
    joblib.dump(val_scores, f"{SCORES_DIR}/{model_name}_val_scores.joblib")
    print(f"  Saved {len(val_scores):,} val score pairs.")

    score_col = f"score_{model_name}"
    print("Scoring Multisocial...")
    multisocial[score_col] = predict_external(
        classifier, multisocial["text"].tolist(), model_name
    )
    print("Scoring AIGTBench...")
    aigtbench[score_col] = predict_external(
        classifier, aigtbench["text"].tolist(), model_name
    )
    print("Scoring Fox8-23...")
    fox8[score_col] = predict_external(
        classifier, fox8["text"].tolist(), model_name
    )

    print(f"Done with {model_name}.")

In [ ]:
score_cols = [f"score_{c['model_name']}" for c in MODEL_CONFIGS]

ms_out   = f"{SCORES_DIR}/multisocial_scores.parquet"
aigt_out = f"{SCORES_DIR}/aigtbench_scores.parquet"
fox8_out = f"{SCORES_DIR}/fox8_scores.parquet"

multisocial[["text", "label", "language", "source"] + score_cols].to_parquet(ms_out, index=True)
aigtbench[["text", "label"] + score_cols].to_parquet(aigt_out, index=True)
fox8[["text", "user_id", "label"] + score_cols].to_parquet(fox8_out, index=True)

print(f"Multisocial scores: {ms_out}")
print(f"AIGTBench scores:   {aigt_out}")
print(f"Fox8-23 scores:     {fox8_out}")

In [ ]:
# results = []

# for config in MODEL_CONFIGS:
#     model_name = config["model_name"]
#     score_col  = f"score_{model_name}"
#     auc_ms = roc_auc_score(multisocial["label"], multisocial[score_col])
#     auc_aigt = roc_auc_score(aigtbench["label"], aigtbench[score_col])
#     auc_fox8_text = roc_auc_score(fox8["label"], fox8[score_col])
#     fox8_agg      = fox8.groupby("user_id")[[score_col, "label"]].mean().reset_index()
#     auc_fox8_user = roc_auc_score(fox8_agg["label"].round().astype(int), fox8_agg[score_col])

#     results.append({
#         "model":         model_name,
#         "multisocial":   round(auc_ms, 4),
#         "aigtbench":     round(auc_aigt, 4),
#         "fox8_text":     round(auc_fox8_text, 4),
#         "fox8_user_agg": round(auc_fox8_user, 4),
#     })

# results_df = pd.DataFrame(results).set_index("model")
# print("ROC-AUC Summary")
# results_df